In [11]:
import pandas as pd

In [12]:
dati_all=pd.read_csv("/Users/carloalbertoalfieri/Desktop/cloud/cloud-image-pipeline/Datav2/images_metadata.csv")
dati_all

,image_key,size_category,difficulty,file_size_kb
0,all_test_images/000000000139.jpg,small,hard,158.02
1,all_test_images/000000000285.jpg,small,easy,327.99
2,all_test_images/000000000632.jpg,small,medium,152.02
3,all_test_images/000000000724.jpg,small,medium,127.06
4,all_test_images/000000000776.jpg,small,easy,172.28
...,...,...,...,...
5061,all_test_images/large_medium_000000033759_15MB...,large,medium,15340.38
5062,all_test_images/large_medium_000000033759_20MB...,large,medium,8832.10
5063,all_test_images/large_medium_000000269113_8MB.jpg,large,medium,7793.44
5064,all_test_images/large_medium_000000269113_15MB...,large,medium,15423.34


In [13]:
dati_small = dati_all[dati_all['size_category'] == "small"]
dati_medium= dati_all[dati_all['size_category'] == "medium"]
dati_large= dati_all[dati_all['size_category'] == "large"]
dati_hard= dati_all[dati_all['difficulty'] == "hard"]

dati_hard_small=dati_hard[dati_hard["size_category"]=="small"]
dati_hard_medium=dati_hard[dati_hard["size_category"]=="medium"]
dati_hard_large=dati_hard[dati_hard["size_category"]=="large"]



In [26]:
dati_hard_medium

,image_key,size_category,difficulty,file_size_kb
225,all_test_images/000000023230.jpg,medium,hard,365.92
549,all_test_images/000000062353.jpg,medium,hard,373.94
1206,all_test_images/000000142238.jpg,medium,hard,353.74
2085,all_test_images/000000239274.jpg,medium,hard,363.90
2404,all_test_images/000000279145.jpg,medium,hard,353.58
3204,all_test_images/000000369812.jpg,medium,hard,418.94
4437,all_test_images/000000514376.jpg,medium,hard,389.34


In [45]:
dati_all.groupby('size_category')
dati_all


,image_key,size_category,difficulty,file_size_kb
0,all_test_images/000000000139.jpg,small,hard,158.02
1,all_test_images/000000000285.jpg,small,easy,327.99
2,all_test_images/000000000632.jpg,small,medium,152.02
3,all_test_images/000000000724.jpg,small,medium,127.06
4,all_test_images/000000000776.jpg,small,easy,172.28
...,...,...,...,...
5061,all_test_images/large_medium_000000033759_15MB...,large,medium,15340.38
5062,all_test_images/large_medium_000000033759_20MB...,large,medium,8832.10
5063,all_test_images/large_medium_000000269113_8MB.jpg,large,medium,7793.44
5064,all_test_images/large_medium_000000269113_15MB...,large,medium,15423.34


In [14]:
import pandas as pd
import numpy as np

# Define the desired proportions
proportions = {'hard': 0.3, 'medium': 0.4, 'easy': 0.3}

def resample_group(group):
    """
    For a single size_category group, resample rows so that
    difficulty distribution matches proportions.
    """
    total = 1000 # keep same total count as original group
    # Compute target counts
    target_counts = {diff: int(round(prop * total)) for diff, prop in proportions.items()}
    # Adjust rounding to make sum exactly total
    diff_sum = sum(target_counts.values())
    if diff_sum != total:
        # Adjust the largest category (or hard) to fix the sum
        target_counts['hard'] += (total - diff_sum)
    
    # Sample with replacement from each difficulty
    sampled_dfs = []
    for diff, n in target_counts.items():
        sub = group[group['difficulty'] == diff]
        if len(sub) == 0:
            # If no rows exist for this difficulty, you cannot sample
            # You could duplicate from another difficulty, but that would break proportions.
            # Better to raise an error or fill with a default.
            raise ValueError(f"No rows with difficulty '{diff}' in size category '{group['size_category'].iloc[0]}'")
        # Sample with replacement (replace=True allows duplication)
        sampled = sub.sample(n=n, replace=True, random_state=42)
        sampled_dfs.append(sampled)
    
    # Combine and shuffle
    result = pd.concat(sampled_dfs, ignore_index=True)
    return result.sample(frac=1, random_state=42)  # shuffle

# Apply to each size_category
dati_resampled = dati_all.groupby('size_category').apply(resample_group)

# Check the new distribution per category
print(dati_resampled.groupby(['size_category', 'difficulty']).size())

size_category  difficulty
large          easy          300
               hard          300
               medium        400
medium         easy          300
               hard          300
               medium        400
small          easy          300
               hard          300
               medium        400
dtype: int64


In [17]:
dati_resampled

image_key  \
size_category                                                          
large         521  all_test_images/large_medium_000000033759_20MB...   
              737   all_test_images/large_easy_000000466835_15MB.jpg   
              740    all_test_images/large_easy_000000203639_8MB.jpg   
              660  all_test_images/large_medium_000000460967_15MB...   
              411  all_test_images/large_medium_000000177065_20MB...   
...                                                              ...   
small         106                   all_test_images/000000439593.jpg   
              270                   all_test_images/000000338304.jpg   
              860                   all_test_images/000000258883.jpg   
              435                   all_test_images/000000340272.jpg   
              102                   all_test_images/000000153527.jpg   

                  difficulty  file_size_kb  
size_category                               
large         521     medium       8832.10  
              737       easy      14226.29  
              740       easy       7836.80  
              660     medium      14759.01  
              411     medium      19632.28  
...                      ...           ...  
small         106       hard        165.78  
              270       hard        110.76  
              860       easy        120.42  
              435     medium        268.77  
              102       hard        198.94  

[3000 rows x 3 columns]

In [15]:
df2 = dati_resampled.reset_index()

# Remove the prefix from image_key
df2['image_key'] = df2['image_key'].str.removeprefix('all_test_images/')
df2=df2.drop("level_1", axis=1)
df2.drop_duplicates(inplace=True)
# Now save the CSV
df2.to_csv('dati_resampled.csv', index=False)

In [20]:
print(df2)

     size_category                           image_key difficulty  \
0            large  large_medium_000000033759_20MB.jpg     medium   
1            large    large_easy_000000466835_15MB.jpg       easy   
2            large     large_easy_000000203639_8MB.jpg       easy   
3            large  large_medium_000000460967_15MB.jpg     medium   
4            large  large_medium_000000177065_20MB.jpg     medium   
...            ...                                 ...        ...   
2993         small                    000000195045.jpg       easy   
2994         small                    000000414510.jpg       hard   
2997         small                    000000258883.jpg       easy   
2998         small                    000000340272.jpg     medium   
2999         small                    000000153527.jpg       hard   

      file_size_kb  
0          8832.10  
1         14226.29  
2          7836.80  
3         14759.01  
4         19632.28  
...            ...  
2993        207.49  
299

In [55]:
df_clean = dati_resampled.reset_index()
df_clean.to_csv('dati_resampled.csv', index=False)

In [21]:
import os
import shutil

# Reset index to make iteration and CSV saving cleaner.
# Since group_keys=True added a MultiIndex, let's reset it.
# Option A: drop old index completely.
df = df2.reset_index(drop=True)
# Option B: keep group key as a column.
# df = dati_resampled.reset_index()

# Create target directory
os.makedirs('final images', exist_ok=True)

# Iterate and copy
for idx, row in df.iterrows():
    src_partial = row['image_key']
    src = os.path.join("all_test_images", str(src_partial))
    # Check if source exists, handle gracefully
    if not os.path.exists(src):
        print(f"Warning: Source file {src} not found. Skipping.")
        continue
    
    # Get base name and create a unique destination name using index
    filename = os.path.basename(src)
    # Option: split name and extension to insert idx
    #name, ext = os.path.splitext(filename)
    #new_filename = f"{name}_{idx}{ext}"
    dst = os.path.join('final images', filename)
    
    shutil.copy2(src, dst)  # copy2 preserves metadata

print("Files copied successfully.")



Files copied successfully.


,image_key,difficulty,file_size_kb
0,all_test_images/large_medium_000000033759_20MB...,medium,8832.10
1,all_test_images/large_easy_000000466835_15MB.jpg,easy,14226.29
2,all_test_images/large_easy_000000203639_8MB.jpg,easy,7836.80
3,all_test_images/large_medium_000000460967_15MB...,medium,14759.01
4,all_test_images/large_medium_000000177065_20MB...,medium,19632.28
...,...,...,...
2995,all_test_images/000000439593.jpg,hard,165.78
2996,all_test_images/000000338304.jpg,hard,110.76
2997,all_test_images/000000258883.jpg,easy,120.42
2998,all_test_images/000000340272.jpg,medium,268.77
